In [1]:
# %% 0. Import & load data
from pathlib import Path

import duckdb
import polars as pl

PROJECT_ROOT = Path.cwd().resolve()
for p in [PROJECT_ROOT, *PROJECT_ROOT.parents]:
    if (p / 'data' / 'raw').exists():
        PROJECT_ROOT = p
        break

RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
OUTPUT_PATH = PROJECT_ROOT / 'data' / 'processed' / 'iged_bersih.parquet'

kandidat = sorted(RAW_DIR.glob('IGED*'))
if not kandidat:
    raise FileNotFoundError(f'Tidak ada file IGED* di {RAW_DIR}. Isi folder: {list(RAW_DIR.iterdir())}')

DATA_PATH = kandidat[0]
print('Membaca:', DATA_PATH)

if DATA_PATH.suffix.lower() == '.parquet':
    df = pl.read_parquet(DATA_PATH)
elif DATA_PATH.suffix.lower() == '.csv':
    df = pl.read_csv(DATA_PATH, infer_schema_length=0)
else:
    raise ValueError(f'Format tidak didukung: {DATA_PATH.suffix}')

print(f'Ukuran awal: {df.shape[0]:,} baris x {df.shape[1]} kolom')
print('Kolom:', df.columns)

WAJIB = {'src', 'trg', 'category', 'split'}
hilang = WAJIB - set(df.columns)
if hilang:
    raise ValueError(f'Kolom tidak ditemukan: {hilang}. Kolom yang ada: {df.columns}')
df.head()

Membaca: /home/jovyan/work/data/raw/IGED_stratified_dataset.csv
Ukuran awal: 1,340,241 baris x 4 kolom
Kolom: ['src', 'trg', 'category', 'split']


src,trg,category,split
str,str,str,str
"""Ketika menunjuk titik dan diat…","""Ketika menunjuk titik dan diat…","""sintaksis_frasa""","""train"""
"""Untuk itu, pihak Daker Madinah…","""Untuk itu, pihak Daker Madinah…","""diksi""","""train"""
"""Laporan Wartawan Tribunnews.co…","""Laporan Wartawan Tribunnews.co…","""preposisi""","""train"""
"""Zambia merupakan salah satu ne…","""Zambia merupakan salah satu ne…","""ambigu""","""train"""
"""Busquets yang-yang memang-mema…","""Busquets yang memang selalu me…","""reduplikasi""","""train"""


In [2]:
# %% 1. Profiling dengan DuckDB SUMMARIZE
connection = duckdb.connect()
connection.register('dataset', df)

profil = connection.execute('SUMMARIZE SELECT * FROM dataset').pl()
profil

column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
str,str,str,str,i64,i32,i32,i32,i32,i32,i64,"decimal[9,2]"
"""src""","""VARCHAR""","""! ! ! "" akhir - akhir ini kita…","""𝝅 = Keuntungan usaha pengolaha…",1207508,null,null,null,null,null,1340241,0.00
"""trg""","""VARCHAR""",""" disimpulkan bahwa H diterim…","""𝝅 = Keuntungan usaha pengolaha…",1229430,null,null,null,null,null,1340241,0.00
"""category""","""VARCHAR""","""afiksasi""","""sintaksis_frasa""",12,null,null,null,null,null,1340241,0.00
"""split""","""VARCHAR""","""test""","""valid""",3,null,null,null,null,null,1340241,0.00


In [3]:
# %% 2. Deteksi missing values
jumlah_baris = len(df)
null_summary = (
    df.null_count()
    .transpose(include_header=True, header_name='kolom', column_names=['jumlah_null'])
    .with_columns(
        (pl.col('jumlah_null') / jumlah_baris * 100).round(2).alias('persen_null')
    )
    .filter(pl.col('jumlah_null') > 0)
    .sort('jumlah_null', descending=True)
)
print('Null per kolom (kosong berarti tidak ada null):')
display(null_summary)

print('String kosong / hanya spasi:')
display(df.select([(pl.col(c).str.strip_chars() == '').sum().alias(c) for c in ['src', 'trg']]))

print('Distribusi kategori (nilai mentah, sebelum dipetakan):')
display(df['category'].value_counts().sort('count', descending=True))

print('Distribusi split:')
display(df['split'].value_counts())

Null per kolom (kosong berarti tidak ada null):


kolom,jumlah_null,persen_null
str,u32,f64


String kosong / hanya spasi:


src,trg
u32,u32
0,0


Distribusi kategori (nilai mentah, sebelum dipetakan):


category,count
str,u32
"""afiksasi""",263218
"""sintaksis_frasa""",252608
"""preposisi""",167735
"""pembentukan_kata""",163725
"""kelengkapan_kalimat""",158766
…,…
"""ambigu""",66037
"""pleonasme""",27175
"""pleonasme_claude""",3500


Distribusi split:


split,count
str,u32
"""valid""",134024
"""test""",134025
"""train""",1072192


In [4]:
# %% 3. Penanganan missing values + normalisasi teks
TEKS = ['src', 'trg']

CATEGORY_MAP = {
    'struktur_frasa': 'sintaksis_frasa',
    'bentuk_kata': 'pembentukan_kata',
    'ambiguitas': 'ambigu',
}
KATEGORI_SAH = [
    'afiksasi', 'pembentukan_kata', 'reduplikasi',
    'sintaksis_frasa', 'preposisi', 'kelengkapan_kalimat',
    'diksi', 'ambigu', 'pleonasme',
]


def normalisasi(kolom: str) -> pl.Expr:
    return (
        pl.col(kolom).cast(pl.Utf8)
        .str.replace_all(r'[\u200b\u200c\u200d\u2060\ufeff]', '')
        .str.replace_all(r'[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]', '')
        .str.replace_all(r'\s+', ' ')
        .str.strip_chars()
        .alias(kolom)
    )


n_category_null = df['category'].null_count()

df_missing_handled = (
    df.with_columns([normalisasi(c) for c in TEKS])
    .with_columns([
        pl.col('category').fill_null('tidak_diketahui').str.strip_chars().str.to_lowercase()
          .replace(CATEGORY_MAP),
        pl.col('split').str.strip_chars().str.to_lowercase()
          .replace({'validation': 'val', 'dev': 'val'}),
    ])
    .filter(
        pl.col('src').is_not_null() & (pl.col('src') != '')
        & pl.col('trg').is_not_null() & (pl.col('trg') != '')
        & pl.col('split').is_not_null()
    )
)

print(f'Null sebelum: {df.null_count().sum_horizontal().sum():,}')
print(f'Null sesudah: {df_missing_handled.null_count().sum_horizontal().sum():,}')
print(f'Category null diisi "tidak_diketahui": {n_category_null:,}')
print(f'Baris dibuang: {len(df) - len(df_missing_handled):,}')
print('Distribusi kategori setelah pemetaan (harus 9 kategori sah):')
display(df_missing_handled['category'].value_counts().sort('count', descending=True))


Null sebelum: 0
Null sesudah: 0
Category null diisi "tidak_diketahui": 0
Baris dibuang: 0
Distribusi kategori setelah pemetaan (harus 9 kategori sah):


category,count
str,u32
"""afiksasi""",263218
"""sintaksis_frasa""",252608
"""preposisi""",167735
"""pembentukan_kata""",163725
"""kelengkapan_kalimat""",158766
…,…
"""ambigu""",66037
"""pleonasme""",27175
"""pleonasme_claude""",3500


In [5]:
# %% 4. Deteksi & penanganan outlier (IQR pada panjang teks)
FAKTOR_IQR = 3.0
KOLOM_NUMERIK = ['src_len', 'trg_len']

df_fitur = df_missing_handled.with_columns([
    pl.col('src').str.len_chars().alias('src_len'),
    pl.col('trg').str.len_chars().alias('trg_len'),
    pl.col('trg').str.split(' ').list.len().alias('trg_kata'),
])

batas_bawah, batas_atas = {}, {}
for kol in KOLOM_NUMERIK:
    q1 = df_fitur[kol].quantile(0.25)
    q3 = df_fitur[kol].quantile(0.75)
    iqr = q3 - q1
    batas_bawah[kol] = max(q1 - FAKTOR_IQR * iqr, 1)
    batas_atas[kol] = q3 + FAKTOR_IQR * iqr
    print(f'{kol}: Q1={q1}, Q3={q3}, IQR={iqr}, '
          f'batas bawah={batas_bawah[kol]:.1f}, batas atas={batas_atas[kol]:.1f}')

df_outlier_handled = df_fitur.filter(
    pl.col('src_len').is_between(batas_bawah['src_len'], batas_atas['src_len'])
    & pl.col('trg_len').is_between(batas_bawah['trg_len'], batas_atas['trg_len'])
    & (pl.col('trg_kata') >= 2)
)
n_outlier = len(df_fitur) - len(df_outlier_handled)
print(f'Outlier dibuang: {n_outlier:,}')
display(df_outlier_handled.select(['src_len', 'trg_len', 'trg_kata']).describe())

# Cek src yang sangat pendek (hanya dilaporkan)
MIN_SRC_PENDEK = 5
src_pendek = df_outlier_handled.filter(pl.col('src_len') < MIN_SRC_PENDEK)
print(f'src < {MIN_SRC_PENDEK} karakter (masih lolos, hanya dilaporkan): {len(src_pendek):,}')
display(src_pendek.select(['src', 'trg', 'category']).head(10))


src_len: Q1=87.0, Q3=189.0, IQR=102.0, batas bawah=1.0, batas atas=495.0
trg_len: Q1=86.0, Q3=185.0, IQR=99.0, batas bawah=1.0, batas atas=482.0
Outlier dibuang: 26,281


statistic,src_len,trg_len,trg_kata
str,f64,f64,f64
"""count""",1.31396e6,1.31396e6,1.31396e6
"""null_count""",0.0,0.0,0.0
"""mean""",144.621218,142.422116,19.588264
"""std""",80.54296,79.309369,10.685431
"""min""",1.0,8.0,3.0
"""25%""",86.0,85.0,12.0
"""50%""",127.0,125.0,17.0
"""75%""",184.0,181.0,25.0
"""max""",495.0,482.0,147.0


src < 5 karakter (masih lolos, hanya dilaporkan): 17


src,trg,category
str,str,str
"""8 .""","""Minahasa Selatan 8.""","""kelengkapan_kalimat"""
"""2 .""","""Aneka gorengan 2.""","""kelengkapan_kalimat"""
"""39 .""","""Milka Kalakamabin 39.""","""kelengkapan_kalimat"""
"""Misi""","""Misi ”Mewujudkan jaringan ekos…","""pleonasme_claude"""
"""8 .""","""Jawa Timur 8.""","""kelengkapan_kalimat"""
"""r""","""Adapun korelasi dapat juga dis…","""pleonasme_claude"""
"""Ya .""","""Ini penumpang ya.""","""kelengkapan_kalimat"""
"""17 .""","""Grup Mega 17.""","""kelengkapan_kalimat"""
"""15 .""","""Kopda Dani 15.""","""kelengkapan_kalimat"""


In [6]:
# %% 5. Deduplikasi
KOLOM_KUNCI = ['src', 'trg']

total_baris = len(df_outlier_handled)
tanpa_koreksi = df_outlier_handled.filter(pl.col('src') == pl.col('trg')).height
baris_unik = df_outlier_handled.unique(subset=KOLOM_KUNCI).height
print(f'Total baris: {total_baris:,}')
print(f'Pasangan tanpa koreksi (src == trg): {tanpa_koreksi:,}')
print(f'Duplikat berdasarkan {KOLOM_KUNCI}: {total_baris - baris_unik:,}')

# 5a. Buang pasangan tanpa koreksi
df_beda = df_outlier_handled.filter(pl.col('src') != pl.col('trg'))

# 5b. Buang duplikat (src, trg)
df_unik = df_beda.unique(subset=KOLOM_KUNCI, keep='first', maintain_order=True)
n_duplikat = len(df_beda) - len(df_unik)

# 5c. Laporkan src dengan >1 target berbeda (tidak dibuang)
konflik_df = (
    df_unik.group_by('src')
    .agg(pl.col('trg').n_unique().alias('n_trg'))
    .filter(pl.col('n_trg') > 1)
)
konflik = konflik_df.height
print(f'src dengan >1 target berbeda (hanya dilaporkan): {konflik:,}')
print('Contoh konflik untuk dicek manual:')
display(
    df_unik.join(konflik_df.select('src'), on='src', how='inner')
    .sort('src')
    .select(['src', 'trg', 'category', 'split'])
    .head(10)
)

# 5d. Buang kebocoran train -> val/test
src_heldout = (
    df_unik.filter(pl.col('split').is_in(['val', 'test']))
    .select('src').unique()
    .with_columns(pl.lit(True).alias('_bocor'))
)
df_dedup = (
    df_unik.join(src_heldout, on='src', how='left')
    .filter(~((pl.col('split') == 'train') & pl.col('_bocor').fill_null(False)))
    .drop('_bocor')
)
n_bocor = len(df_unik) - len(df_dedup)
print(f'Train yang bocor ke val/test dibuang: {n_bocor:,}')


Total baris: 1,313,960
Pasangan tanpa koreksi (src == trg): 5,278
Duplikat berdasarkan ['src', 'trg']: 404
src dengan >1 target berbeda (hanya dilaporkan): 61
Contoh konflik untuk dicek manual:


src,trg,category,split
str,str,str,str
""""" sekali ( jadwal penerbangan …","""""Sekali (jadwal penerbangan) t…","""kelengkapan_kalimat""","""train"""
""""" sekali ( jadwal penerbangan …","""""Sekali (jadwal penerbangan) t…","""kelengkapan_kalimat""","""test"""
"""( - ) kuning negative ( - ) .""","""G2 Negative (-) Kuning Negativ…","""kelengkapan_kalimat""","""train"""
"""( - ) kuning negative ( - ) .""","""D2 Negative (-) Kuning Negativ…","""kelengkapan_kalimat""","""train"""
"""( baca : kpk , gubernur sumut …","""(Baca: Dipanggil KPK, Gubernur…","""kelengkapan_kalimat""","""train"""
"""( baca : kpk , gubernur sumut …","""(baca: Dipanggil KPK, Gubernur…","""kelengkapan_kalimat""","""train"""
""", "" katanya .""","""Untuk menindaklanjutinya,"" kat…","""kelengkapan_kalimat""","""train"""
""", "" katanya .""","""Intinya antisipasi,"" katanya.""","""kelengkapan_kalimat""","""train"""
""", "" katanya .""","""Kan begitu,"" katanya.""","""kelengkapan_kalimat""","""train"""


Train yang bocor ke val/test dibuang: 8


In [7]:
# %% 6. Validasi data dasar
connection.register('dataset_bersih', df_dedup)

validasi = connection.execute(f"""
    SELECT
        COUNT(*) AS total_baris,
        SUM(CASE WHEN src IS NULL OR TRIM(src) = '' THEN 1 ELSE 0 END) AS pelanggaran_src_kosong,
        SUM(CASE WHEN trg IS NULL OR TRIM(trg) = '' THEN 1 ELSE 0 END) AS pelanggaran_trg_kosong,
        SUM(CASE WHEN src = trg THEN 1 ELSE 0 END) AS pelanggaran_src_sama_trg,
        SUM(CASE WHEN split NOT IN ('train', 'val', 'test') THEN 1 ELSE 0 END) AS pelanggaran_split,
        SUM(CASE WHEN src_len < {batas_bawah['src_len']} OR src_len > {batas_atas['src_len']}
                   OR trg_len < {batas_bawah['trg_len']} OR trg_len > {batas_atas['trg_len']}
            THEN 1 ELSE 0 END) AS pelanggaran_batas
    FROM dataset_bersih
""").pl()
display(validasi)

kebocoran = connection.execute("""
    SELECT COUNT(*) AS train_bocor_ke_val_test
    FROM dataset_bersih
    WHERE split = 'train'
      AND src IN (SELECT src FROM dataset_bersih WHERE split IN ('val', 'test'))
""").pl()
display(kebocoran)

kategori_tidak_sah = df_dedup.filter(~pl.col('category').is_in(KATEGORI_SAH))
print(f'Baris dengan kategori di luar 9 kategori sah: {len(kategori_tidak_sah):,}')
if len(kategori_tidak_sah) > 0:
    display(kategori_tidak_sah['category'].value_counts().sort('count', descending=True))

display(
    connection.execute("""
        SELECT split, category, COUNT(*) AS jumlah
        FROM dataset_bersih
        GROUP BY split, category
        ORDER BY split, jumlah DESC
    """).pl()
)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

total_baris,pelanggaran_src_kosong,pelanggaran_trg_kosong,pelanggaran_src_sama_trg,pelanggaran_split,pelanggaran_batas
i64,"decimal[38,0]","decimal[38,0]","decimal[38,0]","decimal[38,0]","decimal[38,0]"
1308276,0,0,0,130875,0


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

train_bocor_ke_val_test
i64
0


Baris dengan kategori di luar 9 kategori sah: 10,411


category,count
str,u32
"""pleonasme_claude""",3478
"""ambigu_claude""",3469
"""diksi_claude""",3464


split,category,jumlah
str,str,i64
"""test""","""afiksasi""",25718
"""test""","""sintaksis_frasa""",24823
"""test""","""preposisi""",16179
"""test""","""pembentukan_kata""",16071
"""test""","""kelengkapan_kalimat""",15574
…,…,…
"""valid""","""ambigu""",6155
"""valid""","""pleonasme""",2671
"""valid""","""pleonasme_claude""",348


In [9]:
# %% 7. Simpan data bersih
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
df_dedup.write_parquet(OUTPUT_PATH)

print(f'Baris sebelum cleaning: {len(df):,}')
print(f'Baris sesudah cleaning: {len(df_dedup):,}')
print(f'Data bersih tersimpan di: {OUTPUT_PATH}')


Baris sebelum cleaning: 1,340,241
Baris sesudah cleaning: 1,308,276
Data bersih tersimpan di: /home/jovyan/work/data/processed/iged_bersih.parquet


In [10]:
# %% 8. Log perubahan baris per tahap
log = pl.DataFrame({
    'tahap': [
        'Data awal',
        'Missing values + normalisasi',
        'Outlier (IQR + min 2 kata)',
        'Buang src == trg',
        'Buang duplikat (src, trg)',
        'Buang kebocoran train -> val/test',
    ],
    'baris': [
        len(df), len(df_missing_handled), len(df_outlier_handled),
        len(df_beda), len(df_unik), len(df_dedup),
    ],
}).with_columns(
    (-pl.col('baris').diff()).alias('baris_dibuang')
).with_columns(
    (pl.col('baris_dibuang') / len(df) * 100).round(3).alias('persen_dari_awal')
)
display(log)

total_dibuang = len(df) - len(df_dedup)
print(f'Total dibuang: {total_dibuang:,} baris ({total_dibuang / len(df) * 100:.2f}% dari data awal)')


tahap,baris,baris_dibuang,persen_dari_awal
str,i64,i64,f64
"""Data awal""",1340241,null,null
"""Missing values + normalisasi""",1340241,0,0.0
"""Outlier (IQR + min 2 kata)""",1313960,26281,1.961
"""Buang src == trg""",1308682,5278,0.394
"""Buang duplikat (src, trg)""",1308284,398,0.03
"""Buang kebocoran train -> val/t…",1308276,8,0.001


Total dibuang: 31,965 baris (2.39% dari data awal)


In [11]:
# %% 9. Ringkasan masalah kualitas data (otomatis)
n_missing_dibuang = len(df) - len(df_missing_handled)

masalah = pl.DataFrame({
    'dimensi_kualitas': [
        'Completeness (baris kosong dibuang)',
        'Validity (outlier panjang / target < 2 kata)',
        'Validity (src == trg, tanpa koreksi)',
        'Uniqueness (duplikat src+trg)',
        'Consistency (kebocoran train -> val/test)',
    ],
    'jumlah_baris': [n_missing_dibuang, n_outlier, tanpa_koreksi, n_duplikat, n_bocor],
}).with_columns(
    (pl.col('jumlah_baris') / len(df) * 100).round(3).alias('persen_dari_awal')
).sort('jumlah_baris', descending=True)
display(masalah)

paling_bermasalah = masalah.row(0, named=True)

ringkasan = f"""
## Ringkasan Data Cleaning IGED

**Dimensi kualitas paling bermasalah:** {paling_bermasalah['dimensi_kualitas']},
sebanyak {paling_bermasalah['jumlah_baris']:,} baris ({paling_bermasalah['persen_dari_awal']}% dari data awal).

**Missing values:** teks dinormalisasi (zero-width, karakter kontrol, spasi ganda).
`category` kosong ({n_category_null:,} baris) diisi `tidak_diketahui`, varian nama kategori
dipetakan ke 9 kategori kanonik. Baris dengan `src`/`trg`/`split` kosong dibuang
({n_missing_dibuang:,} baris). Imputasi median/modus tidak dipakai karena kolom utama berupa teks.

**Outlier:** IQR faktor {FAKTOR_IQR} pada panjang karakter `src` dan `trg`
(batas `src_len` {batas_bawah['src_len']:.0f}-{batas_atas['src_len']:.0f},
`trg_len` {batas_bawah['trg_len']:.0f}-{batas_atas['trg_len']:.0f}), ditambah aturan domain
target minimal 2 kata. Total {n_outlier:,} baris dibuang. Faktor 3.0 dipilih agar hanya
outlier ekstrem yang dibuang dan variasi panjang kalimat yang wajar tetap terjaga.

**Deduplikasi:** kunci `{KOLOM_KUNCI}`. Dibuang: {tanpa_koreksi:,} pasangan `src == trg`,
{n_duplikat:,} duplikat, dan {n_bocor:,} baris train yang `src`-nya bocor ke val/test.
Sebanyak {konflik:,} `src` dengan target berbeda hanya dilaporkan, tidak dibuang.

**Jumlah baris:** dari {len(df):,} menjadi {len(df_dedup):,}
(berkurang {total_dibuang:,} atau {total_dibuang / len(df) * 100:.2f}%).

**Aturan validasi:** `src`/`trg` tidak kosong; `src != trg`; `split` in {{train, val, test}};
panjang teks dalam batas IQR; tidak ada kebocoran train -> val/test; `category` hanya
berisi 9 kategori sah ({len(kategori_tidak_sah):,} pelanggaran).

**Sitasi:** Marier et al. (2026). CASTLE: Context-Aware Semantic Transformer with Knowledge
Graph Enhancement for Low-Resource Grammar Correction. Expert Systems With Applications, 299,
130233. https://doi.org/10.1016/j.eswa.2025.130233
"""
print(ringkasan)

dimensi_kualitas,jumlah_baris,persen_dari_awal
str,i64,f64
"""Validity (outlier panjang / ta…",26281,1.961
"""Validity (src == trg, tanpa ko…",5278,0.394
"""Uniqueness (duplikat src+trg)""",398,0.03
"""Consistency (kebocoran train -…",8,0.001
"""Completeness (baris kosong dib…",0,0.0



## Ringkasan Data Cleaning IGED

**Dimensi kualitas paling bermasalah:** Validity (outlier panjang / target < 2 kata),
sebanyak 26,281 baris (1.961% dari data awal).

**Missing values:** teks dinormalisasi (zero-width, karakter kontrol, spasi ganda).
`category` kosong (0 baris) diisi `tidak_diketahui`, varian nama kategori
dipetakan ke 9 kategori kanonik. Baris dengan `src`/`trg`/`split` kosong dibuang
(0 baris). Imputasi median/modus tidak dipakai karena kolom utama berupa teks.

**Outlier:** IQR faktor 3.0 pada panjang karakter `src` dan `trg`
(batas `src_len` 1-495,
`trg_len` 1-482), ditambah aturan domain
target minimal 2 kata. Total 26,281 baris dibuang. Faktor 3.0 dipilih agar hanya
outlier ekstrem yang dibuang dan variasi panjang kalimat yang wajar tetap terjaga.

**Deduplikasi:** kunci `['src', 'trg']`. Dibuang: 5,278 pasangan `src == trg`,
398 duplikat, dan 8 baris train yang `src`-nya bocor ke val/test.
Sebanyak 61 `src` dengan target berbeda hanya dilaporkan, tidak dibuang.